# MODULE 3 — Label Harmonization and Task Definition

**Project:** Strict Cross-Dataset, Subject-Independent Motor-Imagery EEG Classification

## Goal

Freeze a scientifically defensible common label space between:

- BCI Competition IV Dataset 2a
- PhysioNet EEGMMIDB

using the **actual local recordings discovered in Module 2**.

## Primary candidate task

**3-class motor imagery**

- Left
- Right
- Feet

## Critical rule

No label is assigned from a filename or annotation code alone when the
dataset semantics depend on the recording/run.

This module creates a trial-level event manifest and explicitly records:

- dataset
- subject
- run
- session
- raw event
- event onset
- raw event meaning
- motor-imagery status
- execution vs imagery status
- harmonized class
- keep/discard decision
- reason for the decision

No signal preprocessing or model training occurs here.

## Scientific basis

### BCI-IV-2a

The official competition documentation defines:

- `769` = left-hand cue
- `770` = right-hand cue
- `771` = foot cue
- `772` = tongue cue

and identifies other codes as trial/run/quality/eye-movement events.

For the cross-dataset experiment, tongue has no semantic counterpart in EEGMMIDB,
so it is explicitly excluded from the primary common task.

### EEGMMIDB

PhysioNet defines the meanings of `T1` and `T2` according to the run:

- unilateral runs: T1 = left fist, T2 = right fist
- bilateral hand/foot runs: T1 = both fists, T2 = both feet

The imagery runs are the relevant runs for our project:

- R04, R08, R12 = unilateral fist imagery
- R06, R10, R14 = bilateral fist/foot imagery

Therefore the primary EEGMMIDB mapping is:

- R04/R08/R12: T1 → Left, T2 → Right
- R06/R10/R14: T2 → Feet
- T1 in R06/R10/R14 = both fists → excluded

This follows the official run-dependent semantics rather than treating T1/T2
as globally fixed classes.

## Cell 1 — Imports, paths, and Module 2 manifest loading

This cell loads the audit outputs from Module 2.

The notebook prefers the saved Module 2 manifest rather than rediscovering
files in a different way.

In [2]:
# ============================================================
# CELL 1 — IMPORTS + MODULE 2 MANIFEST LOADING
# ============================================================

from __future__ import annotations

import json
import re
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import mne

from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ------------------------------------------------------------
# Project configuration from Modules 1–2
# ------------------------------------------------------------

PROJECT_ROOT = Path("/Users/ashokvarmabevara/Project2")

BCI2A_ROOT = PROJECT_ROOT / "BCI IV-2a"
EEGMMIDB_ROOT = PROJECT_ROOT / "eegmmidb"

OUTPUT_ROOT = PROJECT_ROOT / "cross_dataset_mi_project"
MANIFEST_ROOT = OUTPUT_ROOT / "manifests"

MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

RAW_AUDIT_PATH = MANIFEST_ROOT / "raw_eeg_audit.csv"
FILE_INVENTORY_PATH = MANIFEST_ROOT / "dataset_file_inventory.csv"

assert BCI2A_ROOT.exists(), f"Missing BCI-IV-2a path: {BCI2A_ROOT}"
assert EEGMMIDB_ROOT.exists(), f"Missing EEGMMIDB path: {EEGMMIDB_ROOT}"
assert RAW_AUDIT_PATH.exists(), f"Missing Module 2 output: {RAW_AUDIT_PATH}"

raw_audit_df = pd.read_csv(RAW_AUDIT_PATH)

print("=" * 78)
print("MODULE 3 — LABEL HARMONIZATION")
print("=" * 78)

print("Module 2 raw audit rows:", len(raw_audit_df))
print("Datasets:", sorted(raw_audit_df["dataset"].dropna().unique().tolist()))

print("\nRaw audit columns:")
print(raw_audit_df.columns.tolist())

MODULE 3 — LABEL HARMONIZATION
Module 2 raw audit rows: 1544
Datasets: ['BCI-IV-2a', 'EEGMMIDB']

Raw audit columns:
['dataset', 'absolute_path', 'relative_path', 'filename', 'suffix', 'subject', 'session', 'run', 'recording_id', 'parse_status', 'file_readable', 'read_error', 'reader', 'sfreq_hz', 'n_channels', 'n_times', 'duration_sec', 'channel_names', 'channel_types', 'annotation_count', 'annotation_descriptions', 'meas_date', 'sampling_rates_raw', 'sample_checked', 'sample_nan', 'sample_inf', 'sample_max_abs', 'sample_ptp_median', 'sample_error', 'channel_signature']


## Cell 2 — Formal label and run specifications

The mapping is represented as explicit tables.

Important design choice:

**BCI-IV-2a supervised source data use the `T` GDF recordings.**

The `E` recordings are not used for the primary supervised classification task
because the original competition evaluation recordings do not provide the same
supervised cue labels as the training session. They will remain documented but
excluded from the primary harmonized labeled dataset.

For EEGMMIDB, only the six imagery runs are eligible.

In [3]:
# ============================================================
# CELL 2 — FROZEN RUN / EVENT SPECIFICATIONS
# ============================================================

PRIMARY_CLASSES = ["left", "right", "feet"]

# ------------------------------------------------------------
# BCI-IV-2a
# ------------------------------------------------------------

BCI_EVENT_MAP = {
    769: {
        "raw_meaning": "cue onset left hand",
        "harmonized_class": "left",
        "keep": True,
        "reason": "Primary common MI class",
    },
    770: {
        "raw_meaning": "cue onset right hand",
        "harmonized_class": "right",
        "keep": True,
        "reason": "Primary common MI class",
    },
    771: {
        "raw_meaning": "cue onset foot",
        "harmonized_class": "feet",
        "keep": True,
        "reason": "Primary common MI class",
    },
    772: {
        "raw_meaning": "cue onset tongue",
        "harmonized_class": None,
        "keep": False,
        "reason": "No defensible EEGMMIDB tongue counterpart",
    },
    768: {
        "raw_meaning": "start of trial",
        "harmonized_class": None,
        "keep": False,
        "reason": "Structural event, not a class cue",
    },
    783: {
        "raw_meaning": "cue unknown",
        "harmonized_class": None,
        "keep": False,
        "reason": "Unknown class event",
    },
    1023: {
        "raw_meaning": "rejected trial",
        "harmonized_class": None,
        "keep": False,
        "reason": "Rejected trial",
    },
    1072: {
        "raw_meaning": "eye movements",
        "harmonized_class": None,
        "keep": False,
        "reason": "Artifact/auxiliary event",
    },
    32766: {
        "raw_meaning": "start of a new run",
        "harmonized_class": None,
        "keep": False,
        "reason": "Structural event",
    },
    276: {
        "raw_meaning": "idling EEG eyes open",
        "harmonized_class": None,
        "keep": False,
        "reason": "Baseline, not MI",
    },
    277: {
        "raw_meaning": "idling EEG eyes closed",
        "harmonized_class": None,
        "keep": False,
        "reason": "Baseline, not MI",
    },
}

BCI_PRIMARY_SESSION_CODE = "T"

# ------------------------------------------------------------
# EEGMMIDB
# ------------------------------------------------------------

EEGMMIDB_UNILATERAL_IMAGERY_RUNS = {"R04", "R08", "R12"}
EEGMMIDB_BILATERAL_IMAGERY_RUNS = {"R06", "R10", "R14"}

EEGMMIDB_PRIMARY_RUNS = (
    EEGMMIDB_UNILATERAL_IMAGERY_RUNS
    | EEGMMIDB_BILATERAL_IMAGERY_RUNS
)

EEGMMIDB_RUN_MAPPING = {}

for run in EEGMMIDB_UNILATERAL_IMAGERY_RUNS:
    EEGMMIDB_RUN_MAPPING[run] = {
        "T0": {
            "meaning": "rest",
            "harmonized_class": None,
            "keep": False,
            "reason": "Rest interval, not a motor-imagery class",
        },
        "T1": {
            "meaning": "left fist imagery",
            "harmonized_class": "left",
            "keep": True,
            "reason": "Left-hand semantic counterpart",
        },
        "T2": {
            "meaning": "right fist imagery",
            "harmonized_class": "right",
            "keep": True,
            "reason": "Right-hand semantic counterpart",
        },
    }

for run in EEGMMIDB_BILATERAL_IMAGERY_RUNS:
    EEGMMIDB_RUN_MAPPING[run] = {
        "T0": {
            "meaning": "rest",
            "harmonized_class": None,
            "keep": False,
            "reason": "Rest interval, not a motor-imagery class",
        },
        "T1": {
            "meaning": "both fists imagery",
            "harmonized_class": None,
            "keep": False,
            "reason": "No matching BCI-IV-2a common class",
        },
        "T2": {
            "meaning": "both feet imagery",
            "harmonized_class": "feet",
            "keep": True,
            "reason": "Foot semantic counterpart",
        },
    }

print("Primary classes:", PRIMARY_CLASSES)
print("BCI session used:", BCI_PRIMARY_SESSION_CODE)
print("EEGMMIDB imagery runs:", sorted(EEGMMIDB_PRIMARY_RUNS))

Primary classes: ['left', 'right', 'feet']
BCI session used: T
EEGMMIDB imagery runs: ['R04', 'R06', 'R08', 'R10', 'R12', 'R14']


## Cell 3 — Robust event extraction utilities

This cell reads annotation timing from the actual GDF/EDF recordings.

It does not load the full signal into memory.

Two important safeguards are implemented:

1. Events are extracted from recordings directly rather than reconstructed from
   aggregate counts.

2. Unknown annotation values are never silently assigned a class.

In [4]:
# ============================================================
# CELL 3 — EVENT EXTRACTION UTILITIES
# ============================================================

def parse_numeric_event_code(description: str):
    """
    Convert BCI-IV-2a annotation descriptions such as '769' or
    '769.0' into integer event codes.
    """
    if description is None:
        return None

    text = str(description).strip()

    try:
        value = float(text)
        if np.isfinite(value) and float(value).is_integer():
            return int(value)
    except Exception:
        pass

    return None


def read_annotations_only(path: Path):
    """
    Read annotations from a raw EEG recording without preloading signal data.
    """
    suffix = path.suffix.lower()

    if suffix == ".gdf":
        raw = mne.io.read_raw_gdf(
            str(path),
            preload=False,
            verbose="ERROR",
        )
    elif suffix == ".edf":
        raw = mne.io.read_raw_edf(
            str(path),
            preload=False,
            verbose="ERROR",
        )
    else:
        raise ValueError(f"Unsupported event reader: {suffix}")

    try:
        annotations = raw.annotations.copy()
    finally:
        try:
            raw.close()
        except Exception:
            pass

    return annotations


def annotations_to_dataframe(
    annotations,
    dataset: str,
    subject: str,
    session: str,
    run: str,
    recording_id: str,
    filename: str,
    file_path: str,
):
    rows = []

    for idx, (onset, duration, description) in enumerate(
        zip(
            annotations.onset,
            annotations.duration,
            annotations.description,
        )
    ):
        rows.append({
            "dataset": dataset,
            "subject": subject,
            "session": session,
            "run": run,
            "recording_id": recording_id,
            "filename": filename,
            "absolute_path": file_path,
            "event_index": idx,
            "onset_sec": float(onset),
            "duration_sec": float(duration),
            "raw_description": str(description),
        })

    return pd.DataFrame(rows)

## Cell 4 — Extract and harmonize BCI-IV-2a training-session events

Only the `T` GDF recordings are used for the primary supervised BCI-IV-2a
dataset.

The event codes are mapped using the official BCI-IV-2a event specification.

The resulting table preserves all observed annotations and identifies exactly
which ones become Left/Right/Feet trials.

In [5]:
# ============================================================
# CELL 4 — BCI-IV-2a EVENT EXTRACTION + HARMONIZATION
# ============================================================

bci_files = raw_audit_df[
    (raw_audit_df["dataset"] == "BCI-IV-2a")
    & (raw_audit_df["suffix"].str.lower() == ".gdf")
    & (raw_audit_df["session"].astype(str).str.upper() == BCI_PRIMARY_SESSION_CODE)
].copy()

print("BCI-IV-2a supervised GDF files:", len(bci_files))

bci_event_rows = []
bci_recording_errors = []

for _, row in tqdm(
    bci_files.iterrows(),
    total=len(bci_files),
    desc="BCI-IV-2a event extraction",
):
    path = Path(row["absolute_path"])

    try:
        annotations = read_annotations_only(path)

        temp = annotations_to_dataframe(
            annotations=annotations,
            dataset=row["dataset"],
            subject=row["subject"],
            session=row["session"],
            run=row["run"],
            recording_id=row["recording_id"],
            filename=row["filename"],
            file_path=row["absolute_path"],
        )

        for _, event in temp.iterrows():
            code = parse_numeric_event_code(event["raw_description"])

            info = BCI_EVENT_MAP.get(
                code,
                {
                    "raw_meaning": "unknown event code",
                    "harmonized_class": None,
                    "keep": False,
                    "reason": "Unknown/unmapped annotation",
                },
            )

            bci_event_rows.append({
                **event.to_dict(),
                "raw_event_code": code,
                "raw_event_meaning": info["raw_meaning"],
                "is_primary_mi_event": bool(
                    code in {769, 770, 771}
                ),
                "harmonized_class": info["harmonized_class"],
                "keep_trial": bool(info["keep"]),
                "decision_reason": info["reason"],
            })

    except Exception as exc:
        bci_recording_errors.append({
            "dataset": row["dataset"],
            "subject": row["subject"],
            "run": row["run"],
            "filename": row["filename"],
            "error": repr(exc),
        })

bci_events_df = pd.DataFrame(bci_event_rows)

print("\nBCI-IV-2a extracted annotation rows:", len(bci_events_df))

if len(bci_events_df):
    print("\nObserved event codes:")
    display(
        bci_events_df
        .groupby(["raw_event_code", "raw_event_meaning"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    print("\nPrimary BCI trial counts:")
    display(
        bci_events_df[
            bci_events_df["keep_trial"]
        ]
        .groupby(["harmonized_class"])
        .size()
        .reset_index(name="trials")
    )

print("\nBCI event extraction errors:", len(bci_recording_errors))

BCI-IV-2a supervised GDF files: 9


BCI-IV-2a event extraction:   0%|          | 0/9 [00:00<?, ?it/s]


BCI-IV-2a extracted annotation rows: 5552

Observed event codes:


,raw_event_code,raw_event_meaning,count
2,768,start of trial,2592
3,769,cue onset left hand,648
4,770,cue onset right hand,648
5,771,cue onset foot,648
6,772,cue onset tongue,648
7,1023,rejected trial,264
9,32766,start of a new run,79
8,1072,eye movements,9
0,276,idling EEG eyes open,8
1,277,idling EEG eyes closed,8



Primary BCI trial counts:


,harmonized_class,trials
0,feet,648
1,left,648
2,right,648



BCI event extraction errors: 0


## Cell 5 — Extract and harmonize EEGMMIDB imagery runs

The code uses the **run number** to interpret T1/T2.

Only:
- R04, R08, R12
- R06, R10, R14

are eligible.

All other runs are excluded from the primary motor-imagery dataset.

The 128-Hz recordings discovered in Module 2 are not dropped here merely because
their sampling rate differs. Their run/subject identity is retained and their
sampling rate is recorded for Module 5 harmonization.

In [6]:
# ============================================================
# CELL 5 — EEGMMIDB EVENT EXTRACTION + HARMONIZATION
# ============================================================

eegmmidb_files = raw_audit_df[
    (raw_audit_df["dataset"] == "EEGMMIDB")
    & (raw_audit_df["suffix"].str.lower() == ".edf")
].copy()

# Normalize run names to R##
eegmmidb_files["run_norm"] = (
    eegmmidb_files["run"]
    .astype(str)
    .str.upper()
    .str.extract(r"(R\d{2})", expand=False)
)

eegmmidb_candidate_files = eegmmidb_files[
    eegmmidb_files["run_norm"].isin(EEGMMIDB_PRIMARY_RUNS)
].copy()

print("EEGMMIDB total EDF files:", len(eegmmidb_files))
print(
    "EEGMMIDB candidate imagery EDF files:",
    len(eegmmidb_candidate_files),
)

eegmmidb_event_rows = []
eegmmidb_recording_errors = []

for _, row in tqdm(
    eegmmidb_candidate_files.iterrows(),
    total=len(eegmmidb_candidate_files),
    desc="EEGMMIDB imagery event extraction",
):
    path = Path(row["absolute_path"])
    run = row["run_norm"]

    run_mapping = EEGMMIDB_RUN_MAPPING.get(run)

    if run_mapping is None:
        continue

    try:
        annotations = read_annotations_only(path)

        temp = annotations_to_dataframe(
            annotations=annotations,
            dataset=row["dataset"],
            subject=row["subject"],
            session=row["session"],
            run=run,
            recording_id=row["recording_id"],
            filename=row["filename"],
            file_path=row["absolute_path"],
        )

        for _, event in temp.iterrows():
            desc = str(event["raw_description"]).strip().upper()

            info = run_mapping.get(
                desc,
                {
                    "meaning": "unknown event",
                    "harmonized_class": None,
                    "keep": False,
                    "reason": "Unknown annotation in imagery run",
                },
            )

            eegmmidb_event_rows.append({
                **event.to_dict(),
                "raw_event_code": desc,
                "raw_event_meaning": info["meaning"],
                "is_primary_mi_event": desc in {"T1", "T2"},
                "harmonized_class": info["harmonized_class"],
                "keep_trial": bool(info["keep"]),
                "decision_reason": info["reason"],
                "original_sampling_rate_hz": row["sfreq_hz"],
            })

    except Exception as exc:
        eegmmidb_recording_errors.append({
            "dataset": row["dataset"],
            "subject": row["subject"],
            "run": run,
            "filename": row["filename"],
            "error": repr(exc),
        })

eegmmidb_events_df = pd.DataFrame(eegmmidb_event_rows)

print("\nEEGMMIDB extracted annotation rows:", len(eegmmidb_events_df))

if len(eegmmidb_events_df):
    print("\nObserved imagery events:")
    display(
        eegmmidb_events_df
        .groupby(
            ["run", "raw_event_code", "raw_event_meaning"]
        )
        .size()
        .reset_index(name="count")
        .sort_values(
            ["run", "count"],
            ascending=[True, False]
        )
    )

    print("\nPrimary EEGMMIDB trial counts:")
    display(
        eegmmidb_events_df[
            eegmmidb_events_df["keep_trial"]
        ]
        .groupby(["run", "harmonized_class"])
        .size()
        .reset_index(name="trials")
        .sort_values(["run", "harmonized_class"])
    )

print(
    "\nEEGMMIDB event extraction errors:",
    len(eegmmidb_recording_errors),
)

EEGMMIDB total EDF files: 1526
EEGMMIDB candidate imagery EDF files: 654


EEGMMIDB imagery event extraction:   0%|          | 0/654 [00:00<?, ?it/s]


EEGMMIDB extracted annotation rows: 19676

Observed imagery events:


,run,raw_event_code,raw_event_meaning,count
0,R04,T0,rest,1640
1,R04,T1,left fist imagery,830
2,R04,T2,right fist imagery,810
3,R06,T0,rest,1640
4,R06,T1,both fists imagery,825
5,R06,T2,both feet imagery,815
6,R08,T0,rest,1638
7,R08,T1,left fist imagery,825
8,R08,T2,right fist imagery,813
9,R10,T0,rest,1640



Primary EEGMMIDB trial counts:


,run,harmonized_class,trials
0,R04,left,830
1,R04,right,810
2,R06,feet,815
3,R08,left,825
4,R08,right,813
5,R10,feet,820
6,R12,left,825
7,R12,right,815
8,R14,feet,820



EEGMMIDB event extraction errors: 0


## Cell 6 — Combine the harmonized trial/event manifest

This creates the single authoritative Module 3 event manifest.

Every retained row is a candidate trial onset.

Important:
- no EEG samples are extracted;
- no temporal window has been applied;
- no normalization has been performed;
- no channels have been selected;
- no subject has been held out.

Those operations belong to later modules.

In [7]:
# ============================================================
# CELL 6 — COMBINED HARMONIZED EVENT MANIFEST
# ============================================================

event_frames = []

if len(bci_events_df):
    bci_keep = bci_events_df[
        [
            "dataset",
            "subject",
            "session",
            "run",
            "recording_id",
            "filename",
            "absolute_path",
            "event_index",
            "onset_sec",
            "duration_sec",
            "raw_description",
            "raw_event_code",
            "raw_event_meaning",
            "is_primary_mi_event",
            "harmonized_class",
            "keep_trial",
            "decision_reason",
        ]
    ].copy()

    event_frames.append(bci_keep)

if len(eegmmidb_events_df):
    phys_keep = eegmmidb_events_df[
        [
            "dataset",
            "subject",
            "session",
            "run",
            "recording_id",
            "filename",
            "absolute_path",
            "event_index",
            "onset_sec",
            "duration_sec",
            "raw_description",
            "raw_event_code",
            "raw_event_meaning",
            "is_primary_mi_event",
            "harmonized_class",
            "keep_trial",
            "decision_reason",
        ]
    ].copy()

    event_frames.append(phys_keep)

if event_frames:
    harmonized_events_df = pd.concat(
        event_frames,
        ignore_index=True,
    )
else:
    harmonized_events_df = pd.DataFrame()

# Explicit class validation
if len(harmonized_events_df):
    retained = harmonized_events_df[
        harmonized_events_df["keep_trial"]
    ].copy()

    assert set(retained["harmonized_class"].dropna().unique()).issubset(
        set(PRIMARY_CLASSES)
    )

    assert retained["harmonized_class"].notna().all()

print("=" * 78)
print("HARMONIZED EVENT MANIFEST")
print("=" * 78)

print("Total observed annotations:", len(harmonized_events_df))
print(
    "Retained candidate MI trials:",
    int(
        harmonized_events_df["keep_trial"].sum()
    ) if len(harmonized_events_df) else 0,
)

if len(harmonized_events_df):
    print("\nRetained trials by dataset/class:")
    display(
        harmonized_events_df[
            harmonized_events_df["keep_trial"]
        ]
        .groupby(["dataset", "harmonized_class"])
        .size()
        .reset_index(name="trials")
        .sort_values(["dataset", "harmonized_class"])
    )

    print("\nRetained trials by subject:")
    display(
        harmonized_events_df[
            harmonized_events_df["keep_trial"]
        ]
        .groupby(["dataset", "subject", "harmonized_class"])
        .size()
        .reset_index(name="trials")
        .head(100)
    )

HARMONIZED EVENT MANIFEST
Total observed annotations: 25228
Retained candidate MI trials: 9317

Retained trials by dataset/class:


,dataset,harmonized_class,trials
0,BCI-IV-2a,feet,648
1,BCI-IV-2a,left,648
2,BCI-IV-2a,right,648
3,EEGMMIDB,feet,2455
4,EEGMMIDB,left,2480
5,EEGMMIDB,right,2438



Retained trials by subject:


,dataset,subject,harmonized_class,trials
0,BCI-IV-2a,S01,feet,72
1,BCI-IV-2a,S01,left,72
2,BCI-IV-2a,S01,right,72
3,BCI-IV-2a,S02,feet,72
4,BCI-IV-2a,S02,left,72
...,...,...,...,...
95,EEGMMIDB,S023,right,23
96,EEGMMIDB,S024,feet,23
97,EEGMMIDB,S024,left,22
98,EEGMMIDB,S024,right,23


## Cell 7 — Class-balance and coverage analysis

This is the decision gate for the 3-class task.

We evaluate:
- total class counts
- subject coverage
- per-subject class presence
- run coverage
- dataset-specific class balance

The goal is not to maximize the number of samples by adding semantically
incompatible classes.

In [8]:
# ============================================================
# CELL 7 — CLASS BALANCE / SUBJECT COVERAGE
# ============================================================

retained_df = harmonized_events_df[
    harmonized_events_df["keep_trial"]
].copy()

assert len(retained_df) > 0, (
    "No harmonized trials were retained. "
    "Inspect event extraction before proceeding."
)

print("=" * 78)
print("CLASS BALANCE AND COVERAGE")
print("=" * 78)

# ------------------------------------------------------------
# Global class balance
# ------------------------------------------------------------

global_class_counts = (
    retained_df
    .groupby(["dataset", "harmonized_class"])
    .size()
    .reset_index(name="trials")
    .sort_values(["dataset", "harmonized_class"])
)

print("\nGlobal class counts:")
display(global_class_counts)

# ------------------------------------------------------------
# Subject-level class coverage
# ------------------------------------------------------------

subject_class = (
    retained_df
    .groupby(
        ["dataset", "subject", "harmonized_class"]
    )
    .size()
    .reset_index(name="trials")
)

print("\nSubject/class coverage:")
display(subject_class.head(150))

coverage_pivot = (
    subject_class
    .pivot_table(
        index=["dataset", "subject"],
        columns="harmonized_class",
        values="trials",
        fill_value=0,
    )
    .reset_index()
)

for cls in PRIMARY_CLASSES:
    if cls not in coverage_pivot.columns:
        coverage_pivot[cls] = 0

coverage_pivot["all_three_classes_present"] = (
    (coverage_pivot["left"] > 0)
    & (coverage_pivot["right"] > 0)
    & (coverage_pivot["feet"] > 0)
)

print("\nSubjects with all three classes:")
display(
    coverage_pivot[
        ["dataset", "subject", "left", "right", "feet",
         "all_three_classes_present"]
    ]
)

# ------------------------------------------------------------
# Imbalance ratios
# ------------------------------------------------------------

imbalance_rows = []

for dataset_name, sub in retained_df.groupby("dataset"):
    counts = (
        sub["harmonized_class"]
        .value_counts()
        .reindex(PRIMARY_CLASSES, fill_value=0)
    )

    max_count = int(counts.max())
    min_count = int(counts.min())

    imbalance_rows.append({
        "dataset": dataset_name,
        "left": int(counts["left"]),
        "right": int(counts["right"]),
        "feet": int(counts["feet"]),
        "max_min_ratio": (
            max_count / min_count
            if min_count > 0
            else np.inf
        ),
    })

imbalance_df = pd.DataFrame(imbalance_rows)

print("\nClass-imbalance summary:")
display(imbalance_df)

CLASS BALANCE AND COVERAGE

Global class counts:


,dataset,harmonized_class,trials
0,BCI-IV-2a,feet,648
1,BCI-IV-2a,left,648
2,BCI-IV-2a,right,648
3,EEGMMIDB,feet,2455
4,EEGMMIDB,left,2480
5,EEGMMIDB,right,2438



Subject/class coverage:


,dataset,subject,harmonized_class,trials
0,BCI-IV-2a,S01,feet,72
1,BCI-IV-2a,S01,left,72
2,BCI-IV-2a,S01,right,72
3,BCI-IV-2a,S02,feet,72
4,BCI-IV-2a,S02,left,72
...,...,...,...,...
145,EEGMMIDB,S040,left,21
146,EEGMMIDB,S040,right,24
147,EEGMMIDB,S041,feet,22
148,EEGMMIDB,S041,left,24



Subjects with all three classes:


harmonized_class,dataset,subject,left,right,feet,all_three_classes_present
0,BCI-IV-2a,S01,72.0,72.0,72.0,True
1,BCI-IV-2a,S02,72.0,72.0,72.0,True
2,BCI-IV-2a,S03,72.0,72.0,72.0,True
3,BCI-IV-2a,S04,72.0,72.0,72.0,True
4,BCI-IV-2a,S05,72.0,72.0,72.0,True
...,...,...,...,...,...,...
113,EEGMMIDB,S105,23.0,22.0,21.0,True
114,EEGMMIDB,S106,24.0,21.0,22.0,True
115,EEGMMIDB,S107,24.0,21.0,23.0,True
116,EEGMMIDB,S108,22.0,23.0,23.0,True



Class-imbalance summary:


,dataset,left,right,feet,max_min_ratio
0,BCI-IV-2a,648,648,648,1.000000
1,EEGMMIDB,2480,2438,2455,1.017227


## Cell 8 — Compare alternative task formulations

The project specification required that 3-class and 2-class formulations be
evaluated rather than assuming the answer.

This cell compares:

1. Left / Right / Feet
2. Left / Right

The primary choice remains three-class only if every dataset provides reliable
coverage of all three classes in the imagery data.

In [9]:
# ============================================================
# CELL 8 — TASK FORMULATION COMPARISON
# ============================================================

task_comparison_rows = []

for dataset_name, sub in retained_df.groupby("dataset"):
    counts = (
        sub["harmonized_class"]
        .value_counts()
        .reindex(PRIMARY_CLASSES, fill_value=0)
    )

    three_class_total = int(counts.sum())
    binary_total = int(
        counts["left"] + counts["right"]
    )

    subjects_all_three = int(
        coverage_pivot[
            (coverage_pivot["dataset"] == dataset_name)
        ]["all_three_classes_present"].sum()
    )

    subjects_left_right = int(
        (
            coverage_pivot[
                coverage_pivot["dataset"] == dataset_name
            ]["left"].gt(0)
            &
            coverage_pivot[
                coverage_pivot["dataset"] == dataset_name
            ]["right"].gt(0)
        ).sum()
    )

    task_comparison_rows.extend([
        {
            "dataset": dataset_name,
            "task": "Left / Right / Feet",
            "classes": 3,
            "retained_trials": three_class_total,
            "subjects_with_complete_class_coverage": subjects_all_three,
        },
        {
            "dataset": dataset_name,
            "task": "Left / Right",
            "classes": 2,
            "retained_trials": binary_total,
            "subjects_with_complete_class_coverage": subjects_left_right,
        },
    ])

task_comparison_df = pd.DataFrame(task_comparison_rows)

print("=" * 78)
print("TASK FORMULATION COMPARISON")
print("=" * 78)

display(task_comparison_df)

primary_task_is_valid = (
    retained_df["harmonized_class"]
    .isin(PRIMARY_CLASSES)
    .all()
    and retained_df[
        retained_df["harmonized_class"] == "feet"
    ].shape[0] > 0
)

if primary_task_is_valid:
    RECOMMENDED_TASK = "3-class: Left / Right / Feet"
else:
    RECOMMENDED_TASK = "2-class: Left / Right"

print("\nRecommended primary task:", RECOMMENDED_TASK)

TASK FORMULATION COMPARISON


,dataset,task,classes,retained_trials,subjects_with_complete_class_coverage
0,BCI-IV-2a,Left / Right / Feet,3,1944,9
1,BCI-IV-2a,Left / Right,2,1296,9
2,EEGMMIDB,Left / Right / Feet,3,7373,109
3,EEGMMIDB,Left / Right,2,4918,109



Recommended primary task: 3-class: Left / Right / Feet


## Cell 9 — Explicit exclusion analysis

This cell makes the discarded categories auditable.

For publication, this prevents statements such as "we simply ignored the other
events."

We explicitly document:
- BCI tongue
- BCI structural/auxiliary events
- BCI evaluation-session recordings
- EEGMMIDB baseline runs
- EEGMMIDB movement-execution runs
- EEGMMIDB both-fist imagery events
- unknown events

In [10]:
# ============================================================
# CELL 9 — EXCLUSION / DISCARDED DATA AUDIT
# ============================================================

exclusion_rows = []

# BCI exclusions from observed annotations
if len(bci_events_df):
    excluded_bci = bci_events_df[
        ~bci_events_df["keep_trial"]
    ].copy()

    if len(excluded_bci):
        bci_excl_summary = (
            excluded_bci
            .groupby(
                ["raw_event_code", "raw_event_meaning", "decision_reason"]
            )
            .size()
            .reset_index(name="events")
        )

        for _, row in bci_excl_summary.iterrows():
            exclusion_rows.append({
                "dataset": "BCI-IV-2a",
                "unit": "event",
                "source": row["raw_event_code"],
                "meaning": row["raw_event_meaning"],
                "excluded_events": int(row["events"]),
                "reason": row["decision_reason"],
            })

# BCI E-session recordings
bci_e_files = raw_audit_df[
    (raw_audit_df["dataset"] == "BCI-IV-2a")
    & (raw_audit_df["suffix"].str.lower() == ".gdf")
    & (raw_audit_df["session"].astype(str).str.upper() == "E")
]

if len(bci_e_files):
    exclusion_rows.append({
        "dataset": "BCI-IV-2a",
        "unit": "recording",
        "source": "E-session GDF",
        "meaning": "competition evaluation session",
        "excluded_events": int(len(bci_e_files)),
        "reason": (
            "No equivalent supervised target labels are assumed for the "
            "primary custom classification protocol."
        ),
    })

# EEGMMIDB run exclusions
eeg_all = raw_audit_df[
    (raw_audit_df["dataset"] == "EEGMMIDB")
    & (raw_audit_df["suffix"].str.lower() == ".edf")
].copy()

eeg_all["run_norm"] = (
    eeg_all["run"]
    .astype(str)
    .str.upper()
    .str.extract(r"(R\d{2})", expand=False)
)

for run, sub in eeg_all.groupby("run_norm"):
    if run not in EEGMMIDB_PRIMARY_RUNS:
        exclusion_rows.append({
            "dataset": "EEGMMIDB",
            "unit": "recording",
            "source": run,
            "meaning": "baseline or motor-execution run",
            "excluded_events": int(len(sub)),
            "reason": (
                "Primary project is strict motor-imagery classification; "
                "baseline and execution runs are excluded."
            ),
        })

if len(eegmmidb_events_df):
    both_fist = eegmmidb_events_df[
        (eegmmidb_events_df["run"].isin(
            EEGMMIDB_BILATERAL_IMAGERY_RUNS
        ))
        & (eegmmidb_events_df["raw_event_code"] == "T1")
    ]

    if len(both_fist):
        exclusion_rows.append({
            "dataset": "EEGMMIDB",
            "unit": "event",
            "source": "T1 in R06/R10/R14",
            "meaning": "both fists imagery",
            "excluded_events": int(len(both_fist)),
            "reason": "No corresponding BCI-IV-2a common class.",
        })

exclusion_df = pd.DataFrame(exclusion_rows)

print("=" * 78)
print("EXCLUSION AUDIT")
print("=" * 78)

display(exclusion_df)

EXCLUSION AUDIT


,dataset,unit,source,meaning,excluded_events,reason
0,BCI-IV-2a,event,276,idling EEG eyes open,8,"Baseline, not MI"
1,BCI-IV-2a,event,277,idling EEG eyes closed,8,"Baseline, not MI"
2,BCI-IV-2a,event,768,start of trial,2592,"Structural event, not a class cue"
3,BCI-IV-2a,event,772,cue onset tongue,648,No defensible EEGMMIDB tongue counterpart
4,BCI-IV-2a,event,1023,rejected trial,264,Rejected trial
5,BCI-IV-2a,event,1072,eye movements,9,Artifact/auxiliary event
6,BCI-IV-2a,event,32766,start of a new run,79,Structural event
7,BCI-IV-2a,recording,E-session GDF,competition evaluation session,9,No equivalent supervised target labels are ass...
8,EEGMMIDB,recording,R01,baseline or motor-execution run,109,Primary project is strict motor-imagery classi...
9,EEGMMIDB,recording,R02,baseline or motor-execution run,109,Primary project is strict motor-imagery classi...


## Cell 10 — Trial-manifest consistency and leakage checks

A trial is defined here by:

**dataset + subject + recording_id + event_index**

This cell ensures:
- no retained trial is duplicated;
- all retained classes belong to the primary task;
- BCI retained events come only from T sessions;
- EEGMMIDB retained events come only from imagery runs;
- no target-subject logic has been introduced.

This is a label-integrity gate, not the final train/test leakage gate.

In [11]:
# ============================================================
# CELL 10 — CONSISTENCY / LABEL-LEAKAGE CHECKS
# ============================================================

retained_key_cols = [
    "dataset",
    "subject",
    "recording_id",
    "event_index",
]

duplicate_trial_rows = retained_df[
    retained_df.duplicated(
        subset=retained_key_cols,
        keep=False,
    )
].copy()

assert len(duplicate_trial_rows) == 0, (
    "Duplicate retained trial identifiers detected."
)

assert set(
    retained_df["harmonized_class"].unique()
).issubset(set(PRIMARY_CLASSES))

# BCI session integrity
bci_retained = retained_df[
    retained_df["dataset"] == "BCI-IV-2a"
]

if len(bci_retained):
    assert (
        bci_retained["session"]
        .astype(str)
        .str.upper()
        .eq("T")
        .all()
    )

    assert (
        bci_retained["raw_event_code"]
        .isin(["769", "770", "771"])
        .all()
        or
        pd.to_numeric(
            bci_retained["raw_event_code"],
            errors="coerce"
        ).isin([769, 770, 771]).all()
    )

# EEGMMIDB run integrity
phys_retained = retained_df[
    retained_df["dataset"] == "EEGMMIDB"
]

if len(phys_retained):
    assert (
        phys_retained["run"]
        .isin(EEGMMIDB_PRIMARY_RUNS)
        .all()
    )

print("=" * 78)
print("MODULE 3 INTERNAL CONSISTENCY CHECK")
print("=" * 78)

print("Retained candidate trials:", len(retained_df))
print("Duplicate retained trials:", len(duplicate_trial_rows))
print("Unique subjects:", retained_df.groupby("dataset")["subject"].nunique().to_dict())
print("Unique recordings:", retained_df.groupby("dataset")["recording_id"].nunique().to_dict())

print("\nLabel integrity: PASS")
print("Dataset/run integrity: PASS")
print("Target-subject leakage introduced by Module 3: NO")

MODULE 3 INTERNAL CONSISTENCY CHECK
Retained candidate trials: 9317
Duplicate retained trials: 0
Unique subjects: {'BCI-IV-2a': 9, 'EEGMMIDB': 109}
Unique recordings: {'BCI-IV-2a': 9, 'EEGMMIDB': 654}

Label integrity: PASS
Dataset/run integrity: PASS
Target-subject leakage introduced by Module 3: NO


## Cell 11 — Save the authoritative Module 3 outputs

The outputs become the inputs for Module 4.

Important files:

- complete observed annotation manifest
- retained candidate-trial manifest
- task-comparison table
- subject/class coverage
- exclusion audit
- machine-readable task specification

In [12]:
# ============================================================
# CELL 11 — SAVE MODULE 3 OUTPUTS
# ============================================================

ALL_EVENTS_PATH = MANIFEST_ROOT / "module_3_all_observed_events.csv"
RETAINED_TRIALS_PATH = MANIFEST_ROOT / "module_3_retained_candidate_trials.csv"
TASK_COMPARISON_PATH = MANIFEST_ROOT / "module_3_task_comparison.csv"
COVERAGE_PATH = MANIFEST_ROOT / "module_3_subject_class_coverage.csv"
EXCLUSION_PATH = MANIFEST_ROOT / "module_3_exclusion_audit.csv"
ERRORS_BCI_PATH = MANIFEST_ROOT / "module_3_bci_event_extraction_errors.csv"
ERRORS_PHYS_PATH = MANIFEST_ROOT / "module_3_eegmmidb_event_extraction_errors.csv"
TASK_SPEC_PATH = MANIFEST_ROOT / "module_3_frozen_task_specification.json"

harmonized_events_df.to_csv(
    ALL_EVENTS_PATH,
    index=False,
)

retained_df.to_csv(
    RETAINED_TRIALS_PATH,
    index=False,
)

task_comparison_df.to_csv(
    TASK_COMPARISON_PATH,
    index=False,
)

coverage_pivot.to_csv(
    COVERAGE_PATH,
    index=False,
)

exclusion_df.to_csv(
    EXCLUSION_PATH,
    index=False,
)

pd.DataFrame(bci_recording_errors).to_csv(
    ERRORS_BCI_PATH,
    index=False,
)

pd.DataFrame(eegmmidb_recording_errors).to_csv(
    ERRORS_PHYS_PATH,
    index=False,
)

task_spec = {
    "module": 3,
    "project": "cross_dataset_subject_independent_mi_eeg",
    "recommended_task": RECOMMENDED_TASK,
    "primary_classes": PRIMARY_CLASSES,
    "bci_iv_2a": {
        "source_file_type": "GDF",
        "supervised_session": "T",
        "event_mapping": {
            "769": "left",
            "770": "right",
            "771": "feet",
        },
        "excluded_event": {
            "772": "tongue",
        },
        "excluded_sessions": ["E"],
    },
    "eegmmidb": {
        "imagery_runs": sorted(EEGMMIDB_PRIMARY_RUNS),
        "unilateral_imagery_runs": sorted(
            EEGMMIDB_UNILATERAL_IMAGERY_RUNS
        ),
        "bilateral_imagery_runs": sorted(
            EEGMMIDB_BILATERAL_IMAGERY_RUNS
        ),
        "mapping": {
            "R04/R08/R12": {
                "T1": "left",
                "T2": "right",
            },
            "R06/R10/R14": {
                "T1": None,
                "T1_reason": "both fists excluded",
                "T2": "feet",
            },
        },
    },
    "important_constraints": [
        "Do not treat T1/T2 as globally fixed meanings across EEGMMIDB runs.",
        "Do not include EEGMMIDB execution runs in the primary MI dataset.",
        "Do not include BCI-IV-2a tongue in the primary cross-dataset task.",
        "Do not use BCI-IV-2a E-session as supervised labels without an explicit validated label source.",
    ],
}

with open(TASK_SPEC_PATH, "w", encoding="utf-8") as f:
    json.dump(task_spec, f, indent=2)

print("=" * 78)
print("MODULE 3 OUTPUTS SAVED")
print("=" * 78)

for path in [
    ALL_EVENTS_PATH,
    RETAINED_TRIALS_PATH,
    TASK_COMPARISON_PATH,
    COVERAGE_PATH,
    EXCLUSION_PATH,
    ERRORS_BCI_PATH,
    ERRORS_PHYS_PATH,
    TASK_SPEC_PATH,
]:
    print(path)

MODULE 3 OUTPUTS SAVED
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_all_observed_events.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_retained_candidate_trials.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_task_comparison.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_subject_class_coverage.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_exclusion_audit.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_bci_event_extraction_errors.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_eegmmidb_event_extraction_errors.csv
/Users/ashokvarmabevara/Project2/cross_dataset_mi_project/manifests/module_3_frozen_task_specification.json


## Cell 12 — Module 3 validation gate

### PASS

All primary classes are semantically valid, retained trials are traceable,
and the task mapping is internally consistent.

### PASS WITH WARNING

The mapping is valid but data imbalance, missing class coverage, or event
extraction errors require attention.

### FAIL

Any retained trial has:
- an invalid class,
- an invalid run,
- duplicate trial identity,
- an unexplained event mapping,
- or an event extraction error affecting the primary task.

Do not continue to Module 4 after FAIL.

In [13]:
# ============================================================
# CELL 12 — MODULE 3 VALIDATION REPORT
# ============================================================

validation = {}

validation["task_is_three_class"] = RECOMMENDED_TASK == "3-class: Left / Right / Feet"
validation["primary_classes_valid"] = set(PRIMARY_CLASSES) == {"left", "right", "feet"}
validation["retained_trials_exist"] = len(retained_df) > 0
validation["no_duplicate_retained_trials"] = len(duplicate_trial_rows) == 0
validation["all_retained_classes_valid"] = set(
    retained_df["harmonized_class"].unique()
).issubset(set(PRIMARY_CLASSES))

validation["bci_session_integrity"] = (
    True
    if len(bci_retained) == 0
    else bci_retained["session"]
        .astype(str)
        .str.upper()
        .eq("T")
        .all()
)

validation["eegmmidb_imagery_run_integrity"] = (
    True
    if len(phys_retained) == 0
    else phys_retained["run"]
        .isin(EEGMMIDB_PRIMARY_RUNS)
        .all()
)

validation["task_spec_saved"] = TASK_SPEC_PATH.exists()
validation["manifest_saved"] = (
    ALL_EVENTS_PATH.exists()
    and RETAINED_TRIALS_PATH.exists()
    and TASK_COMPARISON_PATH.exists()
    and COVERAGE_PATH.exists()
    and EXCLUSION_PATH.exists()
)

critical_checks = [
    validation["task_is_three_class"],
    validation["primary_classes_valid"],
    validation["retained_trials_exist"],
    validation["no_duplicate_retained_trials"],
    validation["all_retained_classes_valid"],
    validation["bci_session_integrity"],
    validation["eegmmidb_imagery_run_integrity"],
    validation["task_spec_saved"],
    validation["manifest_saved"],
]

warnings_list = []

# Check complete three-class coverage at the dataset level
for dataset_name, sub in retained_df.groupby("dataset"):
    counts = (
        sub["harmonized_class"]
        .value_counts()
        .reindex(PRIMARY_CLASSES, fill_value=0)
    )

    for cls in PRIMARY_CLASSES:
        if counts[cls] == 0:
            warnings_list.append(
                f"{dataset_name} contains zero retained {cls} trials."
            )

# Extraction warnings
if len(bci_recording_errors):
    warnings_list.append(
        f"{len(bci_recording_errors)} BCI-IV-2a recordings had event extraction errors."
    )

if len(eegmmidb_recording_errors):
    warnings_list.append(
        f"{len(eegmmidb_recording_errors)} EEGMMIDB recordings had event extraction errors."
    )

if len(coverage_pivot):
    complete_subject_fraction = (
        coverage_pivot["all_three_classes_present"]
        .mean()
    )

    if complete_subject_fraction < 1.0:
        warnings_list.append(
            "Not every subject has all three harmonized classes. "
            "Later modules must handle subject-level class absence explicitly."
        )

module_status = "PASS" if all(critical_checks) else "FAIL"

print("=" * 78)
print("MODULE VALIDATION REPORT — MODULE 3")
print("=" * 78)

for key, value in validation.items():
    print(f"{key:40s}: {value}")

print("\nRecommended task:", RECOMMENDED_TASK)
print("Retained candidate trials:", len(retained_df))

print("\nStatus:", module_status)

if warnings_list:
    print("\nWarnings:")
    for warning in warnings_list:
        print("  -", warning)

if module_status == "PASS" and warnings_list:
    print("\nFINAL MODULE STATUS: PASS WITH WARNING")
elif module_status == "PASS":
    print("\nFINAL MODULE STATUS: PASS")
else:
    print("\nFINAL MODULE STATUS: FAIL")
    print("Do NOT proceed to Module 4.")

MODULE VALIDATION REPORT — MODULE 3
task_is_three_class                     : True
primary_classes_valid                   : True
retained_trials_exist                   : True
no_duplicate_retained_trials            : True
all_retained_classes_valid              : True
bci_session_integrity                   : True
eegmmidb_imagery_run_integrity          : True
task_spec_saved                         : True
manifest_saved                          : True

Recommended task: 3-class: Left / Right / Feet
Retained candidate trials: 9317

Status: PASS

FINAL MODULE STATUS: PASS


# MODULE 3 STOP CONDITION

Review the following before continuing:

1. Exact retained trial counts by dataset and class.
2. Whether Left/Right/Feet are present across subjects.
3. Which EEGMMIDB runs are retained.
4. Which BCI event codes are retained.
5. The exclusion audit.
6. Any event-extraction errors.
7. The exact number of 128-Hz EEGMMIDB recordings among the retained imagery runs.

Module 4 will then perform **channel harmonization by electrode identity and
montage**, not by raw string intersection.